# RAVIR Patch Generation — High-Variance Patch Selection

This utility notebook extracts informative training patches from full-resolution
RAVIR images by selecting patches with the **highest mask variance**.

## Motivation
In retinal images, vessel structures occupy a small fraction of each image.
Training on random patches wastes compute on background-dominated regions.
By selecting patches with high variance in the ground truth mask, we ensure
the training set is enriched with vessel-containing regions.

## Pipeline
1. Split each full-resolution image and mask into 256×256 non-overlapping patches
2. Compute the variance of each **mask** patch (high variance = mix of classes)
3. Select the top-K patches per image (K=30)
4. Save selected patches to a dedicated directory for training

In [ ]:
import os

import cv2
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
IMAGE_DIR = 'data/RAVIR/train/training_images'
MASK_DIR = 'data/RAVIR/train/training_masks'
OUTPUT_IMAGE_DIR = 'data/RAVIR/train/patch_images'
OUTPUT_MASK_DIR = 'data/RAVIR/train/patch_masks'

PATCH_SIZE = 256   # Side length of square patches
TOP_K = 30         # Number of highest-variance patches to keep per image

# Create output directories
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)


def generate_patches(image, mask, patch_size):
    """Split an image and its mask into non-overlapping square patches.

    Only keeps patches that are exactly (patch_size × patch_size),
    discarding partial patches at the image borders.

    Args:
        image: Full-resolution image (H, W) or (H, W, C).
        mask: Corresponding segmentation mask (H, W).
        patch_size: Side length of square patches.

    Returns:
        Tuple of (image_patches, mask_patches) lists.
    """
    patches, mask_patches = [], []
    h, w = image.shape[:2]

    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patch = image[i:i + patch_size, j:j + patch_size]
            mask_patch = mask[i:i + patch_size, j:j + patch_size]

            # Skip partial patches at image borders
            if patch.shape[0] == patch_size and patch.shape[1] == patch_size:
                patches.append(patch)
                mask_patches.append(mask_patch)

    return patches, mask_patches


def calculate_variance(patch):
    """Compute pixel intensity variance of a patch.

    Higher variance in the mask indicates a mix of classes (vessel + background),
    making the patch more informative for training.
    """
    return np.var(patch)


# ── Process all training images ───────────────────────────────────────────────
total_saved = 0

for filename in sorted(os.listdir(IMAGE_DIR)):
    if not filename.endswith('.png'):
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    mask_path = os.path.join(MASK_DIR, filename)

    # Load image and mask
    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Extract all patches
    patches, mask_patches = generate_patches(image, mask, PATCH_SIZE)

    # Rank patches by mask variance (descending) and keep top-K
    variances = [calculate_variance(mp) for mp in mask_patches]
    sorted_indices = np.argsort(variances)[::-1]  # Descending order

    for idx in sorted_indices[:TOP_K]:
        patch_filename = f'{filename[:-4]}_patch_{idx}.png'
        cv2.imwrite(os.path.join(OUTPUT_IMAGE_DIR, patch_filename), patches[idx])
        cv2.imwrite(os.path.join(OUTPUT_MASK_DIR, patch_filename), mask_patches[idx])
        total_saved += 1

print(f'Patch generation completed. Total patches saved: {total_saved}')